# TensiMenu Model 04 — K-Means Clustering + Cluster-based Retrieval

**Author**: Devani

**Pendekatan**:
- Normalisasi: StandardScaler
- Algoritma: **K-Means Clustering** untuk mengelompokkan makanan berdasarkan profil DASH
- Retrieval: Tentukan cluster terdekat untuk user, lalu rank dalam cluster
- Optimal K dipilih via Elbow Method

**Hipotesis**: Makanan punya pola alami (misal sayuran rendah natrium tinggi serat, daging tinggi natrium). Dengan clustering, kita bisa filter cluster yang cocok dengan target user terlebih dahulu, lalu ranking dalam cluster — lebih efisien dan menghasilkan rekomendasi yang lebih *coherent*.

---

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    DRIVE_BASE = '/content/drive/MyDrive/TensiMenu_ML'
    print(f'Google Drive mounted. Base: {DRIVE_BASE}')
except ImportError:
    IN_COLAB = False
    DRIVE_BASE = None
    print('Bukan di Colab — menggunakan path lokal.')

In [ ]:
import json, random, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

ARTIFACTS_DIR = Path(f'{DRIVE_BASE}/artifacts_v4') if IN_COLAB else Path('artifacts_v4')
ARTIFACTS_DIR.mkdir(exist_ok=True)
MODEL_VERSION = '4.0.0-kmeans'
print(f'Model: {MODEL_VERSION}')

In [ ]:
# Loading & preprocessing standar
DATA_PATH = f'{DRIVE_BASE}/datasets/TKPI_2017_dataset  .csv' if IN_COLAB else '../datasets/TKPI_2017_dataset  .csv'
df_raw = pd.read_csv(DATA_PATH)
for col in ['PROTEIN_g', 'KALSIUM_mg', 'NATRIUM_mg']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df = df_raw.rename(columns={
    'KODE': 'food_code', 'NAMA_BAHAN': 'name', 'KATEGORI': 'category',
    'ENERGI_kal': 'energy_kcal', 'LEMAK_g': 'fat_total_g', 'SERAT_g': 'fiber_g',
    'KALSIUM_mg': 'calcium_mg', 'NATRIUM_mg': 'sodium_mg', 'KALIUM_mg': 'potassium_mg',
})

DASH_FEATURES = ['sodium_mg', 'potassium_mg', 'calcium_mg', 'fiber_g', 'fat_total_g']
RELEVANT = ['Serealia', 'Umbi Berpati', 'Kacang & Biji', 'Sayuran',
            'Buah', 'Daging & Unggas', 'Ikan, Kerang & Udang', 'Telur', 'Susu']

df = df[df['category'].isin(RELEVANT)].copy()
df = df[df[DASH_FEATURES].notna().sum(axis=1) >= 3].copy()
for f in DASH_FEATURES:
    df[f] = df.groupby('category')[f].transform(lambda s: s.fillna(s.median()))
    df[f] = df[f].fillna(df[f].median()).clip(lower=0)
df_clean = df.reset_index(drop=True)

X = df_clean[DASH_FEATURES].to_numpy(dtype=np.float64)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Dataset bersih: {len(df_clean)} item')

## 1. Elbow Method untuk Pilih Optimal K

In [ ]:
K_RANGE = range(2, 11)
inertias = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(K_RANGE, inertias, 'o-')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia (WCSS)'); axes[0].set_title('Elbow Method')
axes[1].plot(K_RANGE, silhouettes, 'o-', color='coral')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score'); axes[1].set_title('Silhouette Analysis')
plt.tight_layout(); plt.show()

OPTIMAL_K = K_RANGE[np.argmax(silhouettes)]
print(f'✓ Optimal K (highest silhouette) = {OPTIMAL_K}')

## 2. Final K-Means Model

In [ ]:
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
df_clean['cluster'] = kmeans.fit_predict(X_scaled)

print('=== KARAKTERISTIK SETIAP CLUSTER ===')
cluster_summary = df_clean.groupby('cluster')[DASH_FEATURES].mean().round(1)
cluster_summary['n_items'] = df_clean['cluster'].value_counts().sort_index()
cluster_summary

In [ ]:
# Sample item per cluster
for c in sorted(df_clean['cluster'].unique()):
    samples = df_clean[df_clean['cluster'] == c]['name'].sample(min(5, (df_clean['cluster']==c).sum()), random_state=RANDOM_STATE).tolist()
    print(f'Cluster {c} ({(df_clean["cluster"]==c).sum()} item): {samples}')

## 3. Cluster-based Retrieval

**Strategi**:
1. Transform user target dengan scaler
2. Prediksi cluster terdekat dengan `kmeans.predict()`
3. Filter item hanya dari cluster itu (atau top 2 cluster terdekat)
4. Ranking dalam cluster dengan cosine similarity

In [ ]:
def calculate_personal_targets(profile):
    w, h, age, g = profile['weight_kg'], profile['height_cm'], profile['age'], profile['gender']
    bmr = (10*w) + (6.25*h) - (5*age) + (5 if g == 'laki-laki' else -161)
    t = {'sodium_mg': 2300.0, 'potassium_mg': 4000.0,
         'calcium_mg': 1200.0 if age > 50 else 1000.0,
         'fiber_g': 38.0 if g == 'laki-laki' else 25.0,
         'fat_total_g': round(bmr * 0.27 / 9, 1)}
    if 'ckd' in profile.get('comorbidities', []):
        t['sodium_mg'] = 1500.0; t['potassium_mg'] = 2000.0
    if profile.get('systolic_bp', 0) >= 150:
        t['sodium_mg'] = 1500.0
    return t

def recommend_via_cluster(profile, top_k=15, n_clusters_to_use=2):
    targets = calculate_personal_targets(profile)
    user_vec = np.array([targets[f] for f in DASH_FEATURES])
    user_scaled = scaler.transform(user_vec.reshape(1, -1))
    
    # Jarak ke setiap centroid
    centroid_dists = np.linalg.norm(kmeans.cluster_centers_ - user_scaled, axis=1)
    closest_clusters = np.argsort(centroid_dists)[:n_clusters_to_use]
    
    # Filter item dari cluster terdekat
    candidates = df_clean[df_clean['cluster'].isin(closest_clusters)].copy()
    candidate_indices = candidates.index.tolist()
    
    # Cosine similarity dalam kandidat
    sims = cosine_similarity(user_scaled, X_scaled[candidate_indices])[0]
    candidates['similarity'] = sims
    
    return candidates.nlargest(top_k, 'similarity')[['food_code', 'name', 'category', 'cluster', 'similarity']]

profile = {'gender': 'laki-laki', 'weight_kg': 70, 'height_cm': 170, 'age': 45,
           'comorbidities': [], 'systolic_bp': 140}
recs = recommend_via_cluster(profile, top_k=15)
print('=== TOP 15 (Cluster-based) ===')
recs.reset_index(drop=True)

In [ ]:
joblib.dump(scaler, ARTIFACTS_DIR / 'scaler.pkl')
joblib.dump(kmeans, ARTIFACTS_DIR / 'kmeans_model.pkl')
np.save(ARTIFACTS_DIR / 'item_matrix.npy', X_scaled)
with open(ARTIFACTS_DIR / 'food_ids.json', 'w', encoding='utf-8') as f:
    json.dump(df_clean['food_code'].tolist(), f, ensure_ascii=False)

metadata = {
    'version': MODEL_VERSION,
    'approach': 'K-Means Clustering + Cluster-based Retrieval',
    'trained_at': datetime.utcnow().isoformat() + 'Z',
    'random_state': RANDOM_STATE,
    'n_items': len(df_clean),
    'features': DASH_FEATURES,
    'optimal_k': int(OPTIMAL_K),
    'silhouette_score': float(silhouettes[K_RANGE.index(OPTIMAL_K)]),
    'cluster_centers': kmeans.cluster_centers_.tolist(),
}
with open(ARTIFACTS_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
df_clean.to_csv(ARTIFACTS_DIR / 'food_items_clean.csv', index=False)
print('✓ Artefak v4 tersimpan')